# 2 Design pattern for data generators

In [1]:
import torch

A typical setup for machine learning is that we have an amount of observations, with multiple dimensions. Lets say we have images, size (28, 28) pixels and three colors, so (3, 28, 28).

In [2]:
"""   
torch.rand()    → returns a tensor with random numbers between 0-1 of the given dimension
"""
observations = (50, )
datasize = (3, 28, 28)

dim = observations + datasize

X = torch.rand(dim)
X.shape

torch.Size([50, 3, 28, 28])

To avoid clusters of observations that are highly correlated, we would want to shuffle the data. While we could shuffle the data itself, it is better to use an index and shuffle the index.

This approach is especially useful if your data is too big to fit into your memory. Also take into account the model that runs transformations on your data: it will take up a multitude of the original image.

In that case you would want to feed the model a list of paths to images, files etc, and load a batch of images while training.

In [3]:
"""  
randperm()      → returns a tensor with random permutation of integers from 0 to n - 1
permutation     → different arrangements of the elements of the set / arrangement of a set where the order does not matter
"""
index_list = torch.randperm(len(X))
index_list

tensor([36,  1, 17, 16, 47, 31,  9, 21, 29, 48, 37, 46,  2, 19, 38,  4, 40, 13,
         6,  8, 18, 12,  7, 11, 15, 45, 10, 32, 41, 34, 28, 20, 26, 14, 27, 33,
        35, 49, 44, 42,  0, 39, 24, 25,  3, 43, 30, 23, 22,  5])

With `torch.randperm(n)` we obtain a random permutation of the numbers from 0 to $n$. Next step is using a generator. Simple generators are introduced with [PEP 255](https://www.python.org/dev/peps/pep-0255/)

In [ ]:
"""  
generators      → generator functions allow us to declare a function that behaves like an iterator
next()          → returns the next item in an iteration
"""
gen = (i for i in range(10))
gen

<generator object <genexpr> at 0x7d9116724520>

We can call `next` on a generator:

In [5]:
next(gen)

0

And a second time:

In [6]:
next(gen)

1

This is usefull if we want to generate lists that are infinite:

In [ ]:
"""  
yield           → returns a value from a function
while True      → what exactly needs to be True in this case? According to StackOverflow it means 'loop forever' since True always equals True.
                → tldr: neverending loop
"""

def fib():
    a, b = 0, 1
    while True:
       yield b
       a, b = b, a+b

In [9]:
fibonacci = fib()
for i in range(10):
    print(next(fibonacci))

1
1
2
3
5
8
13
21
34
55


We can use the generator pattern to yield a dataset in batches of observations:

In [10]:
def naive_generator(data, batchsize):
    index_list = torch.randperm(len(data))
    i = 0
    while True:
        index = index_list[i:i+batchsize]
        X = data[index]
        yield X
        i += batchsize

This code will scramble an index, and take a batch-sized chunck of these indices to yield

In [ ]:
"""   
X           → remember X = torch.Size([50, 3, 28, 28]) in en earlier block of code 
"""
gen = naive_generator(X, 32)
for i in range(3):
    batch = next(gen)
    print(f"Shape: {batch.shape}")

Shape: torch.Size([32, 3, 28, 28])
Shape: torch.Size([18, 3, 28, 28])
Shape: torch.Size([0, 3, 28, 28])


However, we run into a problem: after two batches, we ran out of data and we will yield empty tensors...

To solve this problem we will use a nested index:
- `i` loops over the batchsize
- `index` loops through the `index_list`
- every time the `index` would go beyond the datasize, we reset it, and shuffle the `index_list`
- The data is collected in a tensor `X` with the nested `index_list[index]` approach

In [19]:
"""  
data        → X = torch.Size([50, 3, 28, 28])
batchsize   → 32
size        → the amount of observations in X   = 50
shape       → the features per observation in X = [3, 28, 28]
index_list  → random permutation of integers ranging from 0-1 of size 50

X (t.zeros) → shaped (32, 3, 28, 28) = the size of the batches we want 

while loop  → loops through the indexes of batchsize
              if the index is bigger than the amount of observations in X, then the index is reset and a new random permutation of integers is created
              else the index in X (zero) is replaced with a random chosen observation of data
"""

def generator(data, batchsize):
    size = len(data)
    shape = data.shape[1:]
    index_list = torch.randperm(size)
    index = 0

    X = torch.zeros((batchsize, ) + shape)
    while True:

        for i in range(batchsize):
            # i will always run from 0 to batchsize,
            # regardless of how many items you have left
            if index >= size:
                # if your index goes beyond the amount of data
                index = 0
                # we reset it to zero
                index_list = torch.randperm(size)
                # and shuffle the index_list

            # we use the index (that goes from 0 to size)
            # to grab the next (shuffled) index_list item
            # and fill batch i with it
            X[i] = data[index_list[index]]
            index += 1
        yield X

In [20]:
gen = generator(X, 32)
for i in range(3):
    batch = next(gen)
    print(f"Shape: {batch.shape}")

Shape: torch.Size([32, 3, 28, 28])
Shape: torch.Size([32, 3, 28, 28])
Shape: torch.Size([32, 3, 28, 28])


Even though we have only 50 observations, we dont run out of data. We will keep shuffling the data can generate infinite batches, shuffled every time.